In [3]:
import xlwings as xw

# 현재 열린 엑셀 파일의 활성화된 시트를 가져오기
wb = xw.apps.active.books.active
sheet = wb.sheets.active
print('데이터 가져와 연산 수행')
b1 = sheet.range('B1').value
b2 = sheet.range('B2').value
# b1-b2값을 'B3'셀에 넣기
sheet.range('B3').value = b1-b2
print('연산 결과 쓰기 완료')

데이터 가져와 연산 수행
연산 결과 쓰기 완료


In [4]:
import xlwings as xw
import os
# 1. 파일 경로 설정(현재)
file_name = "1-1_xlwings_test.xlsx"
file_path = os.path.join(os.getcwd(), 
                        file_name)
print(file_path)

# 2. 엑셀 열기
wb = xw.Book(file_path)

# 3. 시트 선택
sheet = wb.sheets.active
# 4. 연산
print('데이터 가져와 연산 수행')
b1 = sheet.range('B1').value
b2 = sheet.range('B2').value
# b1-b2값을 'B3'셀에 넣기
sheet.range('B3').value = b1-b2
print('연산 결과 쓰기 완료')

# 저장 및 닫기
wb.save()
wb.close()

C:\ai_x\source\09_generativeAI_RPA\1-1_xlwings_test.xlsx
데이터 가져와 연산 수행
연산 결과 쓰기 완료


In [2]:
import os
from dotenv import load_dotenv
from decouple import config
# 방법1
load_dotenv(".env")
client_id = os.getenv("CLIENT_ID")
print('방법1 :', client_id)
# 방법2
client_id = config('CLIENT_ID')
print('방법2 :', client_id)

방법1 : Et2gBTENKtjTuvgCmoqF
방법2 : Et2gBTENKtjTuvgCmoqF


In [3]:
# 네이버 api TEST (쇼핑검색, 뉴스검색)
import os
import sys
import urllib.request
from dotenv import load_dotenv
load_dotenv()
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
encText = urllib.parse.quote("포켄스")
media = "shop"
url = f"https://openapi.naver.com/v1/search/{media}?sort=date&display=2&query={encText}"

request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Wed, 02 Jul 2025 15:48:35 +0900",
	"total":30880,
	"start":1,
	"display":2,
	"items":[
		{
			"title":"덴탈껌 강아지간식 애견 <b>포켄스<\/b> 덴티 페어리65g S",
			"link":"https:\/\/smartstore.naver.com\/main\/products\/12048797971",
			"image":"https:\/\/shopping-phinf.pstatic.net\/main_8959330\/89593308672.jpg",
			"lprice":"6400",
			"hprice":"",
			"mallName":"로지상회",
			"productId":"89593308672",
			"productType":"2",
			"brand":"",
			"maker":"",
			"category1":"생활\/건강",
			"category2":"반려동물",
			"category3":"강아지 간식",
			"category4":"개껌"
		},
		{
			"title":"애완견비듬 입욕제 독샤워 강아지 시츄 전용 샴푸린스  1개  550ml",
			"link":"https:\/\/link.coupang.com\/re\/PCSNAVERPCSDP?pageKey=8885028690&ctag=8885028690&lptag=P8885028690&itemId=25935600196&vendorItemId=92918956746&spec=10305199",
			"image":"https:\/\/shopping-phinf.pstatic.net\/main_5559697\/55596973002.jpg",
			"lprice":"15850",
			"hprice":"",
			"mallName":"쿠팡",
			"productId":"55596973002",
			"productType":"2",
			"brand":"포켄스",
			"make

In [6]:
# 파일 백업 후, now_list시트를 prev_list로 하고, now_list에 네이버 쇼핑목록 업데이트
from naverOpenai import get_naver_api_data, str_json_dataframe
import xlwings as xw
import handle_sheet as hs

def main():
  # 1. genai_rap.xls 파일 열기
  file_path = "genai_rpa.xlsx"
  wb = xw.Book(file_path)

  # 2~4
  hs.handle_init_sheet(file_path=file_path, wb=wb)
  
  # 5. 네이버 api 쇼핑목록 데이터 출력(json형태str->dict->dataframe)
  str_data = get_naver_api_data("shop", "애완용품")
  df_shopping = str_json_dataframe(str_data)
  
  # 6. 'now_list'시트의 모든 내용을 클리어하고, df_shopping내용('A1'셀)을 업데이트
  hs.update_now_list(wb, df_shopping)

  # 7. 파일 저장 및 닫기
  hs.save_close_file(file_path, wb)

if __name__=='__main__':
  main()

백업 파일 생성 완료 : genai_rpa250702155755.xlsx
prev_list 시트 삭제 완료
now_list 시트 prev_list시트로 복사 완료
now_list 내용 업데이트 완료
workbook 저장 완료


In [5]:
import os
print(os.getcwd())

C:\ai_x\source\09_generativeAI_RPA


In [7]:
import xlwings as xw
from dotenv import load_dotenv
from openai import OpenAI

# 1. excel 열기
wb = xw.Book('genai_rpa.xlsx')

# 2. prev_list, now_list 시트 전체 내용 불러오기
prev_sheet = wb.sheets['prev_list']
now_sheet  = wb.sheets['now_list']
prev_data = prev_sheet.used_range.value # 사용된 범위의 데이터
now_data = now_sheet.used_range.value

# 3. 프롬프트 작성
prompt = f"""다음 두 목록을 비교분석하여, prev_list목록에서 now_list목록으로
바뀐 주요 특징을 추출해 줘.
prev_list 목록 : {prev_data}
now_list 목록 : {now_data}
비교 분석 결과를 바탕으로 구체적인 수치, 상품명, 쇼핑몰명 등을 언급하여 
한글로 100자 이내로 분석글을 작성해 줘."""

# 4. LLM 요청하여 메세지 받기
load_dotenv('.env')
client = OpenAI()
completion = client.chat.completions.create(
  model="gpt-4.1-nano",
  messages=[{"role":"user", "content":prompt}],
  # max_tokens=300
)
# 5. 분석글 출력
print('분석글 :', completion.choices[0].message.content)
print('토근 수 궁금해서 :',completion) 

분석글 : 반려동물 관련 제품에서 강아지 간식·껌 위주에서 목줄·배변용품·용품 등 다양한 품목으로 변화, 몰별 상품도 다양해졌습니다.
토근 수 궁금해서 : ChatCompletion(id='chatcmpl-BoliKKgZ7OJmMyl1NSc6EyyHmMXVA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='반려동물 관련 제품에서 강아지 간식·껌 위주에서 목줄·배변용품·용품 등 다양한 품목으로 변화, 몰별 상품도 다양해졌습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1751439512, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_38343a2f8f', usage=CompletionUsage(completion_tokens=44, prompt_tokens=6015, total_tokens=6059, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [8]:
# 파일 백업 후, now_list시트를 prev_list로 하고, now_list에 네이버 쇼핑목록 업데이트
from naverOpenai import get_naver_api_data, str_json_dataframe
from naverOpenai import get_openai_shopping_analysis, get_openai_news_summarize
import xlwings as xw
import handle_sheet as hs

def main():
  # 1. genai_rap.xls 파일 열기
  file_path = "genai_rpa.xlsx"
  wb = xw.Book(file_path)

  # 2~4
  hs.handle_init_sheet(file_path=file_path, wb=wb)
  
  # 5. 네이버 api 쇼핑목록 데이터 출력(json형태str->dict->dataframe)
  str_data = get_naver_api_data("shop", "포켄스")
  df_shopping = str_json_dataframe(str_data)
  
  # 6. 'now_list'시트의 모든 내용을 클리어하고, df_shopping내용('A1'셀)을 업데이트
  hs.update_now_list(wb, df_shopping)

  # 7. 분석 보고서 업데이트
  # 쇼핑 목록 분석
  result_analysis = get_openai_shopping_analysis(wb)
  # 뉴스 분석
  str_news_data = get_naver_api_data("news", "포켄스") # 뉴스검색 목록
  result_summary = get_openai_news_summarize(str_news_data)
  # 'prev_report'시트 삭제, 'now_report' 시트 복사(prev_report), 분석글 now_report 업데이트
  hs.update_now_report(wb, result_analysis, result_summary)

  # 8. 파일 저장 및 닫기
  hs.save_close_file(file_path, wb)

if __name__=='__main__':
  main()

백업 파일 생성 완료 : genai_rpa250702155856.xlsx
prev_list 시트 삭제 완료
now_list 시트 prev_list시트로 복사 완료
now_list 내용 업데이트 완료
prev_report 시트 삭제 완료
prev_report 시트 복사 완료
pdf 저장완료 : C:\ai_x\source\09_generativeAI_RPA\genai_rpa_250702155901.pdf
workbook 저장 완료


In [10]:
# 파일 백업 후, now_list시트를 prev_list로 하고, now_list에 네이버 쇼핑목록 업데이트
from naverOpenai import get_naver_api_data, str_json_dataframe
from naverOpenai import get_openai_shopping_analysis, get_openai_news_summarize
import xlwings as xw
import handle_sheet as hs

def main():
  # 1. genai_rap.xls 파일 열기
  file_path = "genai_rpa.xlsx"
  wb = xw.Book(file_path)

  # 2~4
  hs.handle_init_sheet(file_path=file_path, wb=wb)
  
  # 5. 네이버 api 쇼핑목록 데이터 출력(json형태str->dict->dataframe)
  str_data = get_naver_api_data("shop", "포켄스")
  df_shopping = str_json_dataframe(str_data)
  
  # 6. 'now_list'시트의 모든 내용을 클리어하고, df_shopping내용('A1'셀)을 업데이트
  hs.update_now_list(wb, df_shopping)

  # 7. 분석 보고서 업데이트
  # 쇼핑 목록 분석
  result_analysis = get_openai_shopping_analysis(wb)
  # 뉴스 분석
  str_news_data = get_naver_api_data("news", "포켄스") # 뉴스검색 목록
  result_summary = get_openai_news_summarize(str_news_data)
  # 'prev_report'시트 삭제, 'now_report' 시트 복사(prev_report), 분석글 now_report 업데이트
  hs.update_now_report(wb, result_analysis, result_summary)

  # 8. 파일 저장 및 닫기
  hs.save_close_file(file_path, wb)

if __name__=='__main__':
  main()

백업 파일 생성 완료 : genai_rpa250702160047.xlsx
prev_list 시트 삭제 완료
now_list 시트 prev_list시트로 복사 완료
now_list 내용 업데이트 완료
prev_report 시트 삭제 완료
prev_report 시트 복사 완료
pdf 저장완료 : C:\ai_x\source\09_generativeAI_RPA\genai_rpa_250702160051.pdf
workbook 저장 완료


In [11]:

import os
import datetime
import shutil # 파일 및 디렉토리 작업 도와주는 lib

def handle_init_sheet(file_path, wb):
  # 2. 백업(파일명 : genai_rap250701125852.xls)
  timestamp = datetime.datetime.now().strftime("%y%m%d%H%M%S")
  backupfile = f"genai_rpa{timestamp}.xlsx"
  shutil.copy(file_path, backupfile)
  print("백업 파일 생성 완료 :", backupfile)
  # 3. 'prev_list' 시트를 삭제
  sheet_names = [s.name for s in wb.sheets]
  if 'prev_list' in sheet_names:
    wb.sheets['prev_list'].delete()
    print('prev_list 시트 삭제 완료')
  else:
    print('prev_list 시트가 존재하지 않아 삭제 못함')

  # 4. 'now_list'시트를 복사하여 'prev_list'시트로 시트 이름 변경
  if 'now_list' in sheet_names:
    now_sheet = wb.sheets['now_list']
    prev_sheet = now_sheet.copy(after=now_sheet)
    prev_sheet.name = 'prev_list'
    print('now_list 시트 prev_list시트로 복사 완료')
  else:
    print('now_list 시트가 존재하지 않아 작업 중단')
    return
  
def update_now_list(wb, df_shopping):
  # 6. 'now_list'시트의 모든 내용을 클리어하고, df_shopping내용('A1'셀)을 업데이트
  now_sheet = wb.sheets['now_list']
  now_sheet.clear()
  now_sheet.range('A1').value = df_shopping
  print("now_list 내용 업데이트 완료")

def save_close_file(file_path, wb):
  # 7. 파일 저장 및 닫기
  wb.save(file_path)
  wb.close()
  print('workbook 저장 완료')

def update_now_report(wb, analysis, summary):
  # 'prev_report'시트 삭제, 'now_report' 시트 복사(prev_report), 분석글 now_report 업데이트
  # 'prev_report'시트 삭제
  sheet_names = [s.name for s in wb.sheets]
  if 'prev_report' in sheet_names:
    wb.sheets['prev_report'].delete()
    print('prev_report 시트 삭제 완료')
  else:
    print('prev_report 시트 존재하지 않아 삭제 못함')
  # 'now_report' 시트 복사(prev_report)
  if 'now_report' in sheet_names:
    now_sheet = wb.sheets['now_report']
    prev_sheet = now_sheet.copy(after=now_sheet)
    prev_sheet.name = 'prev_report'
    print('prev_report 시트 복사 완료')
  else:
    print('now_report 시스트가 없어서 작업을 중단합니다')
    wb.close() # 파일 닫고 작업 중단
    return
  # 분석글(analysis, summary)을 now_report 시트에 업데이트
  current_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  # 시간은 A3셀에 입력
  now_sheet.range('A3').value = current_dt + " 기준" # 오른쪽 정렬
  # now_sheet.range('A3').api.HorizontalAlignment = xw.constants.HAlign.xlHAlignRight
  now_sheet.range('A3').api.HorizontalAlignment = -4152
  # analysis는 A5셀에 입력
  now_sheet.range('A5').value = analysis
  now_sheet.range('A5').api.WrapText = True # 내용이 셀보다 길면 자동 줄바꿈 활성화
  # summary는 A8셀에 입력
  now_sheet.range('A8').value = summary
  now_sheet.range('A8').api.WrapText = True # 줄바꿈 활성화
  # now_sheet를 pdf파일로 생성(genai_rpa_2507011655.pdf)
  timestamp = datetime.datetime.now().strftime("%y%m%d%H%M%S")
  file_name = f"genai_rpa_{timestamp}.pdf"
  file_path = os.path.join(os.getcwd(), file_name)
  now_sheet.api.ExportAsFixedFormat(0, file_path)
                                    #"D:/ai_x/source/09_generativeAI_RPA/"+file_name)
  print('pdf 저장완료 :', file_path)


In [12]:

import os
import sys
import urllib.request
from dotenv import load_dotenv
import json
import pandas as pd
from openai import OpenAI

def get_naver_api_data(media, word):
  "word에 대해 media 검색한 결과의 str을 return"
  load_dotenv()
  client_id = os.getenv("CLIENT_ID")
  client_secret = os.getenv("CLIENT_SECRET")
  encText = urllib.parse.quote(word)
  url = f"https://openapi.naver.com/v1/search/{media}?sort=date&display=20&query={encText}"

  request = urllib.request.Request(url)
  request.add_header("X-Naver-Client-Id",client_id)
  request.add_header("X-Naver-Client-Secret",client_secret)
  response = urllib.request.urlopen(request)
  rescode = response.getcode()
  if(rescode==200):
      response_body = response.read()
      return response_body.decode('utf-8')
  else:
      print("Error Code:" + rescode)

def str_json_dataframe(str_json_result):
  "json스타일의 문자열을 데이터프레임으로 변환하여 return"
  if isinstance(str_json_result, str):
    json_result = json.loads(str_json_result)
  else:
    json_result = {}
  # 딕셔너리 json_result를 데이터 프레임으로 바꿔서 return
  items = json_result.get('items', [])
  df =pd.DataFrame(items)
  df['순위']=range(1, len(df)+1)
  df.set_index("순위", inplace=True)
  return df

def aiconn(prompt):
  # LLM 요청하여 메세지 받기
  load_dotenv('.env')
  client = OpenAI()
  completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"user", "content":prompt}],
    # max_tokens=300
  )
  # 5. 분석글 return
  return completion.choices[0].message.content

def get_openai_shopping_analysis(wb):
  # 2. prev_list, now_list 시트 전체 내용 불러오기
  prev_sheet = wb.sheets['prev_list']
  now_sheet  = wb.sheets['now_list']
  prev_data = prev_sheet.used_range.value # 사용된 범위의 데이터
  now_data = now_sheet.used_range.value

  # 3. 프롬프트 작성
  prompt = f"""다음 두 목록을 비교분석하여, prev_list목록에서 now_list목록으로
  바뀐 주요 특징을 추출해 줘.
  prev_list 목록 : {prev_data}
  now_list 목록 : {now_data}
  비교 분석 결과를 바탕으로 구체적인 수치, 상품명, 쇼핑몰명 등을 언급하여 
  한글로 200자 이내로 분석글을 작성해 줘."""
  return aiconn(prompt=prompt)

def get_openai_news_summarize(str_news_data):
  # 뉴스 요약 return
  prompt = f"""다음 뉴스 내용을 구체적인 수치, 고유명사를 언급하며 글머리를 활용하여 
  한글 200자 이내로 요약 글을 작성해 줘.
  뉴스 내용 {str_news_data}"""
  return aiconn(prompt=prompt)
